# 🗄️ Notebook 01 — Redis Cache (`@cached_node`, TTLs, Fallback)
**Data Governance Copilot · Test Suite**

> **One notebook, one feature.** This notebook isolates and tests the entire caching layer
> from `src/core/cache.py` — with and without a live Redis connection.

---

## What this notebook tests
| # | Test | Description |
|---|------|-------------|
| 1 | Setup & install | Dependencies, env vars |
| 2 | Redis connection | Live connect vs graceful fallback |
| 3 | `make_key()` | SHA-256 determinism & uniqueness |
| 4 | `cache_get / cache_set` | Read / write round-trip (Redis) |
| 5 | In-memory fallback | Full cache ops when Redis is down |
| 6 | TTL enforcement | Value expires after TTL (fakeredis) |
| 7 | `@cached_node` decorator | Sync node caching |
| 8 | `@cached_node` async | Async node caching |
| 9 | `invalidate_pattern()` | Bulk key deletion |
| 10 | Performance benchmark | Cached vs uncached latency |

---

**Requirements:** `redis`, `fakeredis` (for offline/TTL tests), your project on `PYTHONPATH`


## 1 · Install Dependencies

In [ ]:
# Run once — skip if already installed
import subprocess, sys

pkgs = ["redis", "fakeredis"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ Dependencies ready")


## 2 · Imports & Path Setup

In [2]:
import sys, os, time, hashlib, json, asyncio
from pathlib import Path

# ── Point Python at your project root ────────────────────────────────────
# Adjust this path to wherever data-governance-copilot/src lives on your machine
PROJECT_ROOT = Path.cwd().parent  # assumes notebook lives in data-governance-copilot/notebooks/
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"📁 Project root : {PROJECT_ROOT}")
print(f"📂 src on path  : {SRC_PATH}")
print(f"📌 sys.path[0]  : {sys.path[0]}")


📁 Project root : d:\0_PROJECTS\data-governance-copilot\notebook
📂 src on path  : d:\0_PROJECTS\data-governance-copilot\notebook\src
📌 sys.path[0]  : d:\0_PROJECTS\data-governance-copilot\notebook\src


## 3 · Standalone Cache Implementation

The cells below paste a **self-contained copy** of `cache.py` so the notebook runs
even if your project isn't on `PYTHONPATH` yet.
Toggle the import approach with `USE_PROJECT_SOURCE = True/False`.


In [ ]:
USE_PROJECT_SOURCE = False   # ← set True once your src/ is importable

if USE_PROJECT_SOURCE:
    from core.cache import (
        get_client, make_key, cache_get, cache_set,
        invalidate_pattern, cached_node
    )
    from config.settings import AppConfig
    print("✅ Loaded from project source")
else:
    # ── Inline implementation (mirrors src/core/cache.py) ─────────────────
    import hashlib, json, time, asyncio, functools
    from typing import Any, Optional

    _IN_MEMORY: dict = {}   # fallback store

    def get_client(host="localhost", port=6379, password=None, enabled=True):
        """Connect to Redis. Returns None (gracefully) if unavailable or disabled."""
        if not enabled:
            print("⚠️  Redis disabled — using in-memory fallback")
            return None
        try:
            import redis
            client = redis.Redis(
                host=host, port=port, password=password or None,
                decode_responses=True, socket_connect_timeout=2
            )
            client.ping()
            return client
        except Exception as e:
            print(f"⚠️  Redis unavailable ({e}) — using in-memory fallback")
            return None

    def make_key(prefix: str, **kwargs) -> str:
        """Deterministic SHA-256 cache key from prefix + sorted kwargs."""
        payload = json.dumps(kwargs, sort_keys=True, default=str)
        digest = hashlib.sha256(f"{prefix}:{payload}".encode()).hexdigest()
        return f"dgc:{prefix}:{digest[:16]}"

    def cache_get(client, key: str) -> Optional[Any]:
        """Fetch from Redis or in-memory fallback. Returns None on miss."""
        if client:
            val = client.get(key)
            return json.loads(val) if val else None
        return _IN_MEMORY.get(key)

    def cache_set(client, key: str, value: Any, ttl: int = 300) -> bool:
        """Write to Redis (with TTL) or in-memory fallback."""
        if client:
            client.setex(key, ttl, json.dumps(value, default=str))
            return True
        _IN_MEMORY[key] = value   # no TTL in fallback (simplification)
        return True

    def invalidate_pattern(client, pattern: str) -> int:
        """Delete all keys matching glob pattern. Returns count deleted."""
        if client:
            keys = client.keys(pattern)
            if keys:
                return client.delete(*keys)
            return 0
        # in-memory fallback: simple prefix match
        import fnmatch
        to_del = [k for k in _IN_MEMORY if fnmatch.fnmatch(k, pattern)]
        for k in to_del:
            del _IN_MEMORY[k]
        return len(to_del)

    def cached_node(prefix: str, ttl: int = 300):
        """Decorator for sync/async LangGraph node functions with cache."""
        def decorator(fn):
            @functools.wraps(fn)
            def sync_wrapper(state: dict, *args, **kwargs):
                key = make_key(prefix,
                                query=state.get("query", ""),
                                data_products=state.get("data_products", []),
                                time_range=state.get("time_range", ""))
                cached = cache_get(_CLIENT, key)
                if cached is not None:
                    print(f"  🎯 CACHE HIT  [{prefix}] key={key[-8:]}")
                    return cached
                result = fn(state, *args, **kwargs)
                cache_set(_CLIENT, key, result, ttl)
                print(f"  💾 CACHE SET  [{prefix}] key={key[-8:]} ttl={ttl}s")
                return result

            @functools.wraps(fn)
            async def async_wrapper(state: dict, *args, **kwargs):
                key = make_key(prefix,
                                query=state.get("query", ""),
                                data_products=state.get("data_products", []),
                                time_range=state.get("time_range", ""))
                cached = cache_get(_CLIENT, key)
                if cached is not None:
                    print(f"  🎯 CACHE HIT  [{prefix}] key={key[-8:]}  (async)")
                    return cached
                result = await fn(state, *args, **kwargs)
                cache_set(_CLIENT, key, result, ttl)
                print(f"  💾 CACHE SET  [{prefix}] key={key[-8:]} ttl={ttl}s  (async)")
                return result

            return async_wrapper if asyncio.iscoroutinefunction(fn) else sync_wrapper
        return decorator

    _CLIENT = None   # set in the next cell
    print("✅ Inline cache implementation loaded")


## 4 · Test 1 — Redis Connection (Live + Fallback)

In [ ]:
print("=" * 60)
print("TEST 1 — Redis Connection")
print("=" * 60)

# 1a: Try live Redis
print("\n[1a] Connecting to localhost:6379 ...")
_CLIENT = get_client(host="localhost", port=6379, enabled=True)

if _CLIENT:
    info = _CLIENT.info("server")
    print(f"✅ Connected to Redis {info.get('redis_version', 'unknown')}")
else:
    print("ℹ️  No live Redis — fallback active (all subsequent tests still pass)")

# 1b: Force disabled
print("\n[1b] Connecting with enabled=False ...")
disabled = get_client(enabled=False)
assert disabled is None, "Expected None when disabled"
print("✅ Correctly returns None when disabled")


## 5 · Test 2 — `make_key()` Determinism & Uniqueness

In [ ]:
print("=" * 60)
print("TEST 2 — make_key()")
print("=" * 60)

# 2a: Same inputs → same key
k1 = make_key("information_agent", query="show retention", data_products=["retention"], time_range="7d")
k2 = make_key("information_agent", query="show retention", data_products=["retention"], time_range="7d")
assert k1 == k2, "Same inputs must produce same key"
print(f"✅ Deterministic  : {k1}")

# 2b: Different query → different key
k3 = make_key("information_agent", query="show bookings", data_products=["bookings"], time_range="7d")
assert k1 != k3, "Different inputs must produce different key"
print(f"✅ Unique query   : {k3}")

# 2c: Different prefix → different key
k4 = make_key("knowledge_agent", query="show retention", data_products=["retention"], time_range="7d")
assert k1 != k4, "Different prefix must produce different key"
print(f"✅ Unique prefix  : {k4}")

# 2d: kwarg order must not matter
k5 = make_key("information_agent", time_range="7d", data_products=["retention"], query="show retention")
assert k1 == k5, "kwarg order must not affect key"
print(f"✅ Order-stable   : {k5}")

# 2e: Key format check
assert k1.startswith("dgc:information_agent:"), f"Bad prefix format: {k1}"
print(f"\n✅ All make_key() tests passed")


## 6 · Test 3 — `cache_get` / `cache_set` Round-Trip

In [ ]:
print("=" * 60)
print("TEST 3 — cache_get / cache_set round-trip")
print("=" * 60)

test_key = make_key("test_node", query="unit-test", data_products=[], time_range="1d")
test_value = {"result": "hello cache", "score": 0.95, "items": [1, 2, 3]}

# 3a: Miss before write
miss = cache_get(_CLIENT, test_key)
assert miss is None, f"Expected None on cold miss, got: {miss}"
print(f"✅ Cold miss      : None")

# 3b: Write
ok = cache_set(_CLIENT, test_key, test_value, ttl=120)
assert ok, "cache_set must return True"
print(f"✅ Set succeeded  : ttl=120s")

# 3c: Hit after write
hit = cache_get(_CLIENT, test_key)
assert hit == test_value, f"Expected {test_value}, got {hit}"
print(f"✅ Cache hit      : {hit}")

# 3d: Overwrite
new_value = {"result": "updated", "score": 0.99}
cache_set(_CLIENT, test_key, new_value, ttl=60)
hit2 = cache_get(_CLIENT, test_key)
assert hit2 == new_value
print(f"✅ Overwrite OK   : {hit2}")


## 7 · Test 4 — In-Memory Fallback (Redis=None)

In [ ]:
print("=" * 60)
print("TEST 4 — In-memory fallback (client=None)")
print("=" * 60)

_IN_MEMORY.clear()   # start fresh

key_fb = make_key("fallback_test", query="no-redis", data_products=[], time_range="")

# 4a: Miss
assert cache_get(None, key_fb) is None
print("✅ Fallback miss  : None")

# 4b: Set
cache_set(None, key_fb, {"agent": "knowledge", "docs": 3}, ttl=999)
print("✅ Fallback set   : OK (TTL ignored in fallback — by design)")

# 4c: Hit
val = cache_get(None, key_fb)
assert val == {"agent": "knowledge", "docs": 3}
print(f"✅ Fallback hit   : {val}")

# 4d: Independently of live Redis
if _CLIENT:
    live_val = cache_get(_CLIENT, key_fb)
    assert live_val is None, "Fallback writes must not bleed into Redis"
    print("✅ Isolation OK  : fallback write did not bleed into Redis")

print("\n✅ All in-memory fallback tests passed")


## 8 · Test 5 — TTL Enforcement

Uses `fakeredis` so this test works **without a live Redis** and runs in milliseconds.


In [ ]:
print("=" * 60)
print("TEST 5 — TTL enforcement (fakeredis)")
print("=" * 60)

import fakeredis, time

fake = fakeredis.FakeRedis(decode_responses=True)

short_key = make_key("ttl_test", query="expires-fast", data_products=[], time_range="")

# Write with 1-second TTL
cache_set(fake, short_key, {"data": "ephemeral"}, ttl=1)
print("✅ Written with ttl=1s")

# Immediately readable
immediate = cache_get(fake, short_key)
assert immediate == {"data": "ephemeral"}, f"Expected hit, got: {immediate}"
print(f"✅ Immediate hit  : {immediate}")

# Wait for expiry
print("⏳ Waiting 1.5s for TTL to expire ...")
time.sleep(1.5)

expired = cache_get(fake, short_key)
assert expired is None, f"Expected None after TTL, got: {expired}"
print("✅ Post-TTL miss  : None — key expired correctly")

# Verify project TTLs are within sensible ranges
TTL_SPECS = {
    "information_agent":  1800,   # 30 min
    "knowledge_agent":    7200,   # 2 hrs
    "metadata_agent":     3600,   # 1 hr
}
for agent, ttl in TTL_SPECS.items():
    k = make_key(agent, query="ttl-check", data_products=[], time_range="")
    cache_set(fake, k, {"ok": True}, ttl=ttl)
    remaining = fake.ttl(k)
    assert 0 < remaining <= ttl, f"{agent}: TTL out of range — {remaining}"
    print(f"✅ {agent:25s} ttl={ttl:5d}s  redis_ttl={remaining}s")

print("\n✅ All TTL tests passed")


## 9 · Test 6 — `@cached_node` Decorator (Sync)

In [ ]:
print("=" * 60)
print("TEST 6 — @cached_node decorator (sync)")
print("=" * 60)

import fakeredis as _fr
_fake_client = _fr.FakeRedis(decode_responses=True)

# Patch _CLIENT used inside cached_node closure
_original_client = _CLIENT

call_count = 0

@cached_node("information_agent", ttl=1800)
def information_node(state: dict) -> dict:
    global call_count
    call_count += 1
    return {
        "agent_results": [{"agent": "information", "result": f"rows={call_count}"}],
        "sources": ["analytics.retention_metrics"],
    }

# Temporarily wire the decorator to fake redis
_CLIENT = _fake_client

state_a = {"query": "show retention rate", "data_products": ["retention"], "time_range": "7d"}
state_b = {"query": "show bookings",        "data_products": ["bookings"],  "time_range": "30d"}

# First call — cache MISS, fn executes
r1 = information_node(state_a)
assert call_count == 1
print(f"✅ 1st call (miss) : fn executed, call_count={call_count}")

# Second call same state — cache HIT, fn not called
r2 = information_node(state_a)
assert call_count == 1, f"fn should NOT have been called again, count={call_count}"
assert r1 == r2
print(f"✅ 2nd call (hit)  : fn skipped,  call_count={call_count}")

# Different state — new miss
r3 = information_node(state_b)
assert call_count == 2
print(f"✅ 3rd call (miss) : different state, call_count={call_count}")

_CLIENT = _original_client  # restore
print("\n✅ @cached_node sync decorator works correctly")


## 10 · Test 7 — `@cached_node` Decorator (Async)

In [ ]:
print("=" * 60)
print("TEST 7 — @cached_node decorator (async)")
print("=" * 60)

import fakeredis as _fr2
_fake_async_client = _fr2.FakeRedis(decode_responses=True)

async_call_count = 0

@cached_node("knowledge_agent", ttl=7200)
async def knowledge_node(state: dict) -> dict:
    global async_call_count
    async_call_count += 1
    await asyncio.sleep(0.05)   # simulate async I/O
    return {
        "agent_results": [{"agent": "knowledge", "docs": async_call_count}],
        "sources": ["pgvector:governance_policies"],
    }

_CLIENT = _fake_async_client

state_k = {"query": "what is data lineage", "data_products": [], "time_range": ""}

async def run_async_tests():
    global async_call_count
    r1 = await knowledge_node(state_k)
    assert async_call_count == 1
    print(f"✅ 1st await (miss) : executed, count={async_call_count}")

    r2 = await knowledge_node(state_k)
    assert async_call_count == 1, f"Expected no re-execution, got count={async_call_count}"
    assert r1 == r2
    print(f"✅ 2nd await (hit)  : skipped,  count={async_call_count}")
    return True

result = asyncio.run(run_async_tests())
assert result

_CLIENT = _original_client
print("\n✅ @cached_node async decorator works correctly")


## 11 · Test 8 — `invalidate_pattern()` Bulk Deletion

In [ ]:
print("=" * 60)
print("TEST 8 — invalidate_pattern()")
print("=" * 60)

import fakeredis as _fr3
inv_client = _fr3.FakeRedis(decode_responses=True)

# Write several keys
agents = ["information_agent", "knowledge_agent", "metadata_agent"]
states = [
    {"query": f"query {i}", "data_products": [], "time_range": "7d"}
    for i in range(4)
]

written_keys = []
for agent in agents:
    for state in states:
        k = make_key(agent, **state)
        cache_set(inv_client, k, {"agent": agent}, ttl=3600)
        written_keys.append((agent, k))

total_before = len(inv_client.keys("dgc:*"))
print(f"✅ Written {total_before} keys total")

# Invalidate only information_agent keys
deleted = invalidate_pattern(inv_client, "dgc:information_agent:*")
print(f"✅ Deleted {deleted} information_agent keys (expected {len(states)})")
assert deleted == len(states), f"Expected {len(states)} deleted, got {deleted}"

# Remaining keys should all be non-information
remaining = inv_client.keys("dgc:*")
for k in remaining:
    assert "information_agent" not in k, f"Stale key found: {k}"
print(f"✅ {len(remaining)} keys remain — none from information_agent")

# Invalidate everything
deleted_all = invalidate_pattern(inv_client, "dgc:*")
print(f"✅ Bulk wipe: deleted {deleted_all} keys")
assert len(inv_client.keys("dgc:*")) == 0
print("✅ Store is empty after full wipe")

# In-memory fallback invalidation
_IN_MEMORY.clear()
for i in range(3):
    k = make_key("information_agent", query=f"q{i}", data_products=[], time_range="")
    _IN_MEMORY[k] = {"i": i}
del_fb = invalidate_pattern(None, "dgc:information_agent:*")
assert del_fb == 3
assert len(_IN_MEMORY) == 0
print(f"✅ In-memory fallback invalidation: deleted {del_fb} keys")


## 12 · Test 9 — Performance Benchmark (Cached vs Uncached)

In [ ]:
print("=" * 60)
print("TEST 9 — Performance benchmark")
print("=" * 60)

import fakeredis as _fr4, time

bench_client = _fr4.FakeRedis(decode_responses=True)
_CLIENT = bench_client

call_counter = 0

@cached_node("bench_agent", ttl=300)
def slow_node(state: dict) -> dict:
    global call_counter
    call_counter += 1
    time.sleep(0.02)   # simulate 20ms DB query
    return {"result": "computed", "call": call_counter}

bench_state = {"query": "benchmark query", "data_products": ["retention"], "time_range": "7d"}
RUNS = 20

# Cold run (miss)
t0 = time.perf_counter()
slow_node(bench_state)
cold_ms = (time.perf_counter() - t0) * 1000

# Warm runs (hits)
times = []
for _ in range(RUNS):
    t0 = time.perf_counter()
    slow_node(bench_state)
    times.append((time.perf_counter() - t0) * 1000)

warm_avg = sum(times) / len(times)
speedup  = cold_ms / warm_avg if warm_avg > 0 else float("inf")

print(f"  Uncached (1st call) : {cold_ms:.2f} ms")
print(f"  Cached   (avg {RUNS}x) : {warm_avg:.3f} ms")
print(f"  Speedup             : {speedup:.1f}x")
print(f"  Fn call count       : {call_counter}  (should be 1)")
assert call_counter == 1, f"Expected 1 real call, got {call_counter}"
assert warm_avg < cold_ms, "Cached path must be faster"
print(f"\n✅ Cache is {speedup:.0f}x faster — fn executed exactly once")

_CLIENT = _original_client


## 13 · Test Summary

In [ ]:
print("=" * 60)
print("  NOTEBOOK 01 — CACHE LAYER TEST SUMMARY")
print("=" * 60)

results = [
    ("Redis connection (live + disabled)",          "✅"),
    ("make_key() determinism & uniqueness",          "✅"),
    ("cache_get / cache_set round-trip",             "✅"),
    ("In-memory fallback (Redis=None)",              "✅"),
    ("TTL enforcement (fakeredis)",                  "✅"),
    ("@cached_node sync decorator",                  "✅"),
    ("@cached_node async decorator",                 "✅"),
    ("invalidate_pattern() bulk deletion",           "✅"),
    ("Performance benchmark (cached vs uncached)",   "✅"),
]

for desc, status in results:
    print(f"  {status}  {desc}")

print()
print("  TTL reference (from settings.py):")
print("    information_agent : 1800s (30 min) — SQL results")
print("    knowledge_agent   : 7200s (2 hrs)  — RAG documents")
print("    metadata_agent    : 3600s (1 hr)   — Collibra metadata")
print("    capacity/rule     : NOT cached")
print()
print("  All 9 tests PASSED 🎉")
